# 🎨 Neural Gesture Mannequin — Interactive Dual-Mode Inference & Anime Sketch Generation
**Author:** Muhammad Aliffandy (Apple Developer Academy & PukulEnam AI/ML Engineer)  
**Model:** Multi-Task Volumetric PoseNet (Trained on 114,554 COCO Images on NVIDIA DGX H100)  

### 🌟 Complete End-to-End Pipeline:
1. **Input Photo:** Any custom human action photo (uploaded from your computer or URL).
2. **Mode 1: Skeleton Wireframe:** 17 kinematic nodes & Line of Action predicted by your DGX-trained model.
3. **Mode 2: Volumetric 3D Mannequin:** Tapered cylinders, cross-contour perspective ellipses & torso boxes.
4. **Mode 3: Low-Poly Anime Mannequin Sketch:** AI-synthesized faceted anime underdrawing line art directly guided by your model's skeleton rig!

## 🛠️ Step 1: Clone Repository & Install Dependencies

In [ ]:
# 1. Clone repository
!rm -rf hp-tra
!git clone https://github.com/MuhammadAliffandy/hp-tra.git
%cd hp-tra

# 2. Install lightweight dependencies
!pip install -q pycocotools torchvision opencv-python matplotlib Pillow

## ☁️ Step 2: Mount Google Drive & Load Trained Model (`.pth`)

In [ ]:
from google.colab import drive
import os
import torch
from src.model import MultiTaskVolumetricPoseNet

# Mount Drive
drive.mount('/content/drive')

# Model path in your Google Drive (transferred from DGX)
MODEL_PATH = "/content/drive/MyDrive/Neural_Gesture_Mannequin_Models/volumetric_posenet_best.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

# Instantiate Model (MobileNetV3-Large)
model = MultiTaskVolumetricPoseNet(backbone_name="mobilenet_v3_large", pretrained=False).to(device)

if os.path.exists(MODEL_PATH):
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
        val_err = checkpoint.get("val_err", "N/A")
        val_pck = checkpoint.get("val_pck", "N/A")
        print(f"✅ Model successfully loaded from Drive!")
        print(f"   - Best Val Pixel Error : {val_err:.2f} px")
        print(f"   - Best Val PCK@0.1     : {val_pck:.1f} %")
    else:
        model.load_state_dict(checkpoint)
        print("✅ Model state dict loaded successfully!")
else:
    print(f"⚠️ Checkpoint not found at: {MODEL_PATH}")
    print("Please verify that rclone transfer to xxdrive:Neural_Gesture_Mannequin_Models/ has completed.")

model.eval()

## 📸 Step 3: Upload Your Photo (or Provide URL)

In [ ]:
from google.colab import files
from PIL import Image
import urllib.request

# Choose input mode:
# Option A: Upload from your laptop
# Option B: Use an image URL from the web

IMAGE_URL = ""  # Leave empty if you want to upload a file from your computer
TARGET_IMAGE_PATH = "test_input.jpg"

if IMAGE_URL.strip() != "":
    print(f"🌐 Downloading image from URL: {IMAGE_URL}")
    urllib.request.urlretrieve(IMAGE_URL, TARGET_IMAGE_PATH)
    print("✅ Download complete!")
else:
    print("📁 Click 'Choose Files' below to upload your photo:")
    uploaded = files.upload()
    if uploaded:
        TARGET_IMAGE_PATH = list(uploaded.keys())[0]
        print(f"✅ Uploaded: {TARGET_IMAGE_PATH}")
    else:
        print("No file selected, using default sample.")

## 🚀 Step 4: Run DGX Model Inference & Display 4-Panel Master Artist Canvas

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as T
from src.visualize import create_dual_mode_canvas, draw_skeleton_mode, draw_volume_mode

# 1. Preprocess Input Image
transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

raw_pil = Image.open(TARGET_IMAGE_PATH).convert("RGB")
resized_pil = raw_pil.resize((256, 256), Image.Resampling.BILINEAR)
raw_bgr = cv2.cvtColor(np.array(resized_pil), cv2.COLOR_RGB2BGR)

inp_tensor = transform(raw_pil).unsqueeze(0).to(device)

# 2. Forward Pass through Multi-Task Volumetric PoseNet
with torch.no_grad():
    pred_coords, pred_volume, pred_vis_logits = model(inp_tensor)
    
    coords_np = pred_coords[0].cpu().numpy()             # [17, 2] in [0, 1]
    volume_np = pred_volume[0].cpu().numpy()             # [6] in [0, 1]
    vis_np = torch.sigmoid(pred_vis_logits[0]).cpu().numpy() # [17] confidence

# 3. Render Individual Panels
blank_white = np.full((256, 256, 3), 255, dtype=np.uint8)

panel1_photo = np.array(resized_pil) # RGB
panel2_skeleton = cv2.cvtColor(draw_skeleton_mode(raw_bgr, coords_np, vis_np, conf_thresh=0.25), cv2.COLOR_BGR2RGB)
panel3_volume = cv2.cvtColor(draw_volume_mode(blank_white, coords_np, volume_np, vis_np, conf_thresh=0.25), cv2.COLOR_BGR2RGB)
panel4_combined = cv2.cvtColor(draw_volume_mode(blank_white, coords_np, volume_np, vis_np, conf_thresh=0.25), cv2.COLOR_BGR2RGB)
panel4_combined = cv2.cvtColor(draw_skeleton_mode(cv2.cvtColor(panel4_combined, cv2.COLOR_RGB2BGR), coords_np, vis_np, conf_thresh=0.25), cv2.COLOR_BGR2RGB)

# 4. Plot Master 4-Panel Comparison
fig, axes = plt.subplots(1, 4, figsize=(22, 6))

axes[0].imshow(panel1_photo)
axes[0].set_title("[1] Input Photo", fontsize=13, fontweight='bold', pad=10)
axes[0].axis("off")

axes[1].imshow(panel2_skeleton)
axes[1].set_title("[2] Mode 1: Skeleton Wireframe\n(Line of Action & Kinematics)", fontsize=12, fontweight='bold', pad=10)
axes[1].axis("off")

axes[2].imshow(panel3_volume)
axes[2].set_title("[3] Mode 2: 3D Volumetric Mannequin\n(Cylinders & Cross-Contours)", fontsize=12, fontweight='bold', pad=10)
axes[2].axis("off")

axes[3].imshow(panel4_combined)
axes[3].set_title("[4] Combined Artist Canvas\n(Rig + Volumetric Primitives)", fontsize=12, fontweight='bold', pad=10)
axes[3].axis("off")

plt.tight_layout()
output_preview_path = "master_mannequin_preview.png"
plt.savefig(output_preview_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"🎉 SUCCESS! High-res comparison saved to: {output_preview_path}")

## 🎨 Step 5: Joint-Driven 2D Anime Mannequin Sketch (Atari Underdrawing)
Generates an authentic **Manga / Anime Drawing Base (アタリ Underdrawing)** strictly guided by your model's **2D Joint Coordinates & Volumetric Muscle Radii**!
- **100% Anatomical Structure:** Head with facial crosshairs, clavicle V-lines, ribcage oval, pelvic bowl, organic muscle contour curves, and joint pivot rings.
- **Zero Clothes/Wrinkle Noise:** Does not trace photorealistic clothes; produces pure anatomical pose base for digital illustrators.
- **Dual Artist Palettes:** Generates both **Warm Sepia Ink** and **Non-Photo Blue Manga Pencil** styles!

In [ ]:
# ==============================================================================
# 🎨 STEP 5: 2D JOINT-DRIVEN ANIME MANNEQUIN SKETCH (REFINED ORGANIC ATARI)
# 100% Guided by Model 1 2D Joints & Muscle Volumes — Smooth Organic Curves!
# ==============================================================================

import sys
# Ensure freshest module is loaded without stale Colab cache
for k in list(sys.modules.keys()):
    if 'src.visualize' in k:
        del sys.modules[k]

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from src.visualize import generate_joint_driven_anime_sketch

# 1. Input: Predicted 2D Joint Coordinates & Volume Radii from Step 4
print("🎨 Synthesizing Refined Organic Anime Mannequin Sketch from 2D Joints...")

# 2. Generate Warm Sepia Manga Pose Base (RGB: 110, 70, 55)
sketch_sepia = generate_joint_driven_anime_sketch(
    coords_np=coords_np,
    volume_params=volume_np,
    vis_np=vis_np,
    canvas_size=512,
    style="sepia"
)

# 3. Generate Classic Non-Photo Blue Pencil Sketch (RGB: 50, 120, 200)
sketch_blue = generate_joint_driven_anime_sketch(
    coords_np=coords_np,
    volume_params=volume_np,
    vis_np=vis_np,
    canvas_size=512,
    style="blue_pencil"
)

# 4. Plot Master 4-Panel Comparison Showcase
fig, axes = plt.subplots(1, 4, figsize=(22, 6))

# Panel 1: Original Input Photo
axes[0].imshow(panel1_photo)
axes[0].set_title("[1] Input Photo", fontsize=13, fontweight='bold', pad=10)
axes[0].axis("off")

# Panel 2: 2D Skeleton + Volumetric Rig (Our Model)
axes[1].imshow(panel4_combined)
axes[1].set_title("[2] Model 2D Joints & Volume Rig", fontsize=12, fontweight='bold', pad=10)
axes[1].axis("off")

# Panel 3: Refined Sepia Underdrawing
axes[2].imshow(sketch_sepia)
axes[2].set_title("[3] Refined Sepia Anime Mannequin\n(Organic Curves & Tapering)", fontsize=12, fontweight='bold', pad=10)
axes[2].axis("off")

# Panel 4: Refined Non-Photo Blue Pencil Sketch
axes[3].imshow(sketch_blue)
axes[3].set_title("[4] Refined Blue Manga Pencil\n(Organic Gesture Underdrawing)", fontsize=12, fontweight='bold', pad=10)
axes[3].axis("off")

plt.tight_layout()
output_sepia_path = "anime_mannequin_sepia.png"
output_blue_path = "anime_mannequin_blue.png"
Image.fromarray(sketch_sepia).save(output_sepia_path)
Image.fromarray(sketch_blue).save(output_blue_path)
plt.savefig("master_sketch_pipeline.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"🎉 SUCCESS! Refined Organic Anime Base saved to:")
print(f"   - Sepia Style : {output_sepia_path}")
print(f"   - Blue Style  : {output_blue_path}")
print(f"   - Full Canvas : master_sketch_pipeline.png")
